# Step 1

Load Embedding Model

In [1]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer(
  "sentence-transformers/all-MiniLM-L6-v2"
)


c:\Users\ahlaw\OneDrive\Desktop\PYTORCH\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6777.42it/s]


## Step 2

Create Documents

In [2]:
documents = [
    "Dogs are loyal pets",
    "Cats are independent animals",
    "Python is a programming language"
]

## Step 3

Convert To Embeddings

In [ ]:
embeddings = model.encode(documents)
print(embeddings.shape)

# 3 documents
# 384 features each

(3, 384)


## Lesson 2 — Similarity Search

This lesson answers a very important question:

Once we have embeddings, how do we actually find the most relevant documents?

## Step 1

Load Model

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19397.15it/s]


## Step 2

Documents

In [5]:
documents = [
    "Dogs are loyal pets",
    "Cats like sleeping",
    "Python is a programming language",
    "Transformers power modern AI"
]

## Step 3

Generate Embeddings

In [ ]:
doc_embeddings = model.encode(documents)
print(doc_embeddings.shape)

(4, 384)


## Query Embedding

In [7]:
query = "Tell me about puppies"

## CONVERT

In [8]:
query_embedding = model.encode(query)

## Now We Need A Similarity Score

### General Retrieval Algorithm
Query
   ↓
Embedding
   ↓

Compare With Every Document

Doc1 → Score
Doc2 → Score
Doc3 → Score
Doc4 → Score

   ↓

Sort By Score

   ↓

Top K Results

RAG Connection

This is the exact retrieval stage of RAG.

User Question

      ↓
Embedding

      ↓
Similarity Search

      ↓
Top K Chunks

      ↓
LLM

      ↓
Answer

# Lesson 3 — Cosine Similarity In Practice

Until now we know:

Query

   ↓

Embedding

   ↓

Vector

   ↓

Compare Against Documents

   ↓
   
Retrieve Best Matches

But we still don't know:

How exactly do we compare two vectors?

## Step 1

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Step 2

Load model

In [10]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7358.05it/s]


## Step 3

Documents

In [11]:
documents = [
    "Dogs are loyal pets",
    "Cats like sleeping",
    "Python is a programming language",
    "Transformers power modern AI"
]

## Step 4

Create embeddings

In [12]:
doc_embeddings = model.encode(documents)

## Step 5

Query

In [16]:
query = "Tell me about puppies"

query_embedding = model.encode(query)
print(query_embedding.shape)
print(doc_embeddings.shape)

(384,)
(4, 384)


## Step 6

Compute similarity

In [17]:
scores = cosine_similarity(
    [query_embedding],
    doc_embeddings
)

# Output shape:

# (1,4)

# Meaning:

# 1 query
# 4 document scores
print(scores)

[[0.52520746 0.30860853 0.18097961 0.09265582]]


## Retrieve Best Match

In [20]:
import numpy as np

best_idx = np.argmax(scores)
print(best_idx)
print(documents[best_idx])

0
Dogs are loyal pets


Why FAISS Exists

Suppose:

4 documents

Current method:

Compare against all 4

Easy.

Suppose:

10 million documents

Current method:

Compare against all 10 million

Very slow.

FAISS solves:

How can we find nearest vectors
without comparing against everything?

That is the entire purpose of FAISS.

# Lesson 4 — FAISS Fundamentals

What Is FAISS?

FAISS stands for:

Facebook AI Similarity Search

Developed by:

Meta Platforms

Purpose:

Fast similarity search over very large vector collections.

Important Clarification

Many beginners think:

FAISS creates embeddings

Wrong.

FAISS does NOT create embeddings.

Embedding Model:

Sentence Transformer

BGE

E5

OpenAI Embeddings

creates vectors.

FAISS:

Stores vectors
Searches vectors

Nothing more.

Production Pipeline
Documents

     ↓
Embedding Model

     ↓
Vectors

     ↓
FAISS Index

     ↓
Storage

Query:

User Query

      ↓
Embedding Model

      ↓
Query Vector

      ↓
FAISS Search

      ↓
Top K Results

Think of:

Index = Data Structure

that makes searching faster.

Without Index:

Search Every Vector

With Index:

Search Smartly

# First FAISS Program
### Step 1

#### Imports

In [2]:
import faiss
import numpy as np

## Step 2

Create vectors

In [ ]:
vectors = np.array([
    [1.0, 2.0],
    [2.0, 3.0],
    [8.0, 9.0]
], dtype=np.float32)

# Meaning:

# 3 vectors
# 2 dimensions each

## Creating The First Index

In [4]:
dimension = 2

index = faiss.IndexFlatL2(
    dimension
)

What Is IndexFlatL2?

This is the simplest FAISS index.

Meaning:

Flat

↓

Store all vectors.

L2

↓

Use Euclidean Distance.

Internally:

Query
   ↓
Compare Against Every Vector

Wait...

Isn't that brute force?

Yes.

Then why use it?

Because:

Learn FAISS API
Foundation for advanced indexes
Still highly optimized C++ code

## Add Vectors

In [6]:
index.add(vectors)

# Now FAISS stores:

# [1,2]

# [2,3]

# [8,9]

print(index.ntotal)

6


## Search

In [8]:
query = np.array(
    [[1.5, 2.5]],
    dtype=np.float32
)

In [10]:
distances, indices = index.search(
    query,
    k=2
)
print(distances)
print(indices)

[[0.5 0.5]]
[[0 1]]


# Real Embeddings Example

In [11]:
from sentence_transformers import SentenceTransformer

c:\Users\ahlaw\OneDrive\Desktop\PYTORCH\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Documents

In [12]:
documents = [
    "Dogs are loyal pets",
    "Cats like sleeping",
    "Python is a programming language",
    "Transformers power modern AI"
]

## Generate embeddings:

In [13]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(
    documents
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4291.38it/s]


In [14]:
print(embeddings.shape)

(4, 384)


## Create index:

In [17]:
index = faiss.IndexFlatL2(384)


In [18]:
index.add(
    embeddings.astype(np.float32)
)

## Search:

In [19]:
query = "Tell me about puppies"

query_vector = model.encode(
    [query]
).astype(np.float32)

## Retrieve:

In [25]:
distances, indices = index.search(
    query_vector,
    k=2
)
print(distances,indices)
print(documents[indices[0][0]])

[[0.9495852 1.3827829]] [[0 1]]
Dogs are loyal pets


FAISS Limitation

FAISS stores:

Vectors

only.

It does NOT automatically store:

{
    "text": "...",
    "author": "...",
    "source": "...",
    "date": "..."
}

You must manage metadata yourself.

Example:

documents[indices[0][0]]

to recover the text.

This limitation is one reason ChromaDB became popular.

# Lesson 5 — Building First Vector Store

a real application needs:

Store Documents

Store Embeddings

Store Metadata

Search Documents

Return Results


Not just raw vector IDs.

 What Is A Vector Store?

A Vector Store is simply:

Vectors
+
Original Documents
+
Metadata
+
Search Logic

Think:

FAISS
+
Python
=
Simple Vector Store

Designing Our Vector Store

We want:

store.add_documents(...)

and later:

store.search(...)

# Step 1 — Imports

In [26]:
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

## Step 2 — Load Embedding Model

In [27]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5722.18it/s]


## Step 3 — Documents

In [28]:
documents = [
    "Dogs are loyal pets",
    "Cats like sleeping",
    "Python is a programming language",
    "Transformers power modern AI"
]

## Generate Embeddings

In [31]:
embeddings = model.encode(
    documents
)
print(embeddings.shape)

(4, 384)


## Create FAISS Index

In [30]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

## Add Vectors

In [32]:
index.add(
    embeddings.astype(np.float32)
)

But We Have A Problem

Suppose search returns:

[[2]]

What does:

2

mean?

FAISS only knows:

Vector Number 2

It does NOT know:

Python is a programming language

## Storing Documents

We need a mapping.

In [34]:
document_store = {
    0: "Dogs are loyal pets",
    1: "Cats like sleeping",
    2: "Python is a programming language",
    3: "Transformers power modern AI"
}
document_store = documents

## Search Function

In [35]:
def search(query, k=2):

    query_embedding = model.encode(
        [query]
    ).astype(np.float32)

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(
            documents[idx]
        )

    return results

In [37]:
results = search(
    "Tell me about puppies"
)
print(results)

['Dogs are loyal pets', 'Cats like sleeping']


## What ChromaDB Actually Does

A common misconception:

ChromaDB = Magic AI Tool

No.

Internally it provides:

Documents
Metadata
Embeddings
Storage
Search
Persistence
Filtering

Many of the things we are manually building now.

# Lesson 6 — ChromaDB Fundamentals

Mental Model

Instead of:

Vector Search Library

ChromaDB is closer to:

Vector Database

# First ChromaDB Program

## Step 1

Import

In [38]:
import chromadb

## Step 2

Create Client

In [ ]:
client=chromadb.Client()

# Think:

# Client

# means:

# Connection to Database

# Very similar to:

# MongoClient()

## Creating A Collection

In [ ]:
collection = client.create_collection(
    name="my_documents"
)

# Now we have:

# my_documents

# which can store:

# documents
# vectors
# metadata

## Adding Documents

In [41]:
documents = [
    "Dogs are loyal pets",
    "Cats like sleeping",
    "Python is a programming language"
]

## Add:

In [ ]:
collection.add(
    documents=documents,
    ids=["1", "2", "3"]
)

# Wait...

# Where Are The Embeddings?

# We never created them.

# Interesting.

# What ChromaDB Is Doing

# Internally:

# Documents
#      ↓
# Embedding Function
#      ↓
# Vectors
#      ↓
# Storage

# automatically.

# Huge Difference From FAISS

# FAISS:

# embeddings = model.encode(...)
# index.add(embeddings)

# Manual.

# ChromaDB:

# collection.add(...)

# Automatic.

C:\Users\ahlaw\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:19<00:00, 4.37MiB/s]


## Querying

Suppose:

In [ ]:
query = "Tell me about puppies"
results = collection.query(
    query_texts=[
        "Tell me about puppies"
    ],
    n_results=2 # top 2 matches
)
print(results)

# Notice:

# [['1', '2']]

# instead of:

# ['1', '2']

# Why?

# Because ChromaDB supports:

# Multiple Queries At Once

{'ids': [['1', '2']], 'embeddings': None, 'documents': [['Dogs are loyal pets', 'Cats like sleeping']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.9495848417282104, 1.3827831745147705]]}


Persistence

One of ChromaDB's biggest advantages.

FAISS example:

index.add(...)

Program ends.

Everything disappears.

ChromaDB can persist to disk.

client = chromadb.PersistentClient(
    path="./my_db"
)

Now:

Embeddings
Documents
Metadata

are saved.

In [ ]:
client = chromadb.PersistentClient(
    path="./my_db"
)

# Why Persistence Matters

# Imagine:

# 500,000 PDF chunks

# Embedding them takes:

# Hours

# You don't want:

# Restart Program
#       ↓
# Re-embed Everything

# Persistence avoids this.

Real RAG Workflow
PDF

 ↓
Chunking

 ↓
Embeddings

 ↓
ChromaDB

Store once.

Later:

User Query

      ↓
ChromaDB Search

      ↓
Top Chunks

      ↓
LLM

      ↓
Answer

No re-embedding of documents.

# Phase 6 — Lesson 7
ChromaDB Collections

This lesson is less about retrieval algorithms and more about how real systems organize data.

A beginner usually creates:

collection = client.create_collection(
    name="documents"
)

and stores everything there.

This works for:

100 documents

but becomes messy for:

100,000 documents

or

multiple projects

Bad Design

Suppose:

HR Policies
Research Papers
Customer Tickets
Product Manuals

all go into:

collection = "documents"

Then every query searches:

Everything

Not ideal.

## Creating Collections

In [46]:
hr_collection = client.create_collection(
    name="hr_docs"
)

research_collection = client.create_collection(
    name="research_papers"
)

## Get Existing Collection

Suppose collection already exists.

Wrong:

client.create_collection(
    name="hr_docs"
)

again.

In [47]:
collection = client.get_collection(
    name="hr_docs"
)

## Get Or Create

In [ ]:
collection = client.get_or_create_collection(
    name="hr_docs"
)

# Meaning:

# Exists?
#     ↓
# Use It

# Doesn't Exist?
#     ↓
# Create It

# Extremely common in production code.

## Adding Documents

In [49]:
collection.add(
    documents=[
        "Employees get 20 annual leaves",
        "Medical insurance is provided"
    ],
    ids=[
        "doc1",
        "doc2"
    ]
)

## Counting Documents

In [50]:
collection.count()

2

## Viewing Collection Data

In [51]:
collection.get()

{'ids': ['doc1', 'doc2'],
 'embeddings': None,
 'documents': ['Employees get 20 annual leaves',
  'Medical insurance is provided'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None, None]}

## Retrieve Specific Documents

In [52]:
collection.get(
    ids=["doc1"]
)

{'ids': ['doc1'],
 'embeddings': None,
 'documents': ['Employees get 20 annual leaves'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None]}

## Updating Documents

In [53]:
collection.update(
    ids=["doc1"],
    documents=[
        "Employees get 25 annual leaves"
    ]
)

## Deleting Documents

In [54]:
collection.delete(
    ids=["doc1"]
)

{'deleted': 1}

## Delete Entire Collection

In [55]:
client.delete_collection(
    name="hr_docs"
)

## Collection Metadata

In [ ]:
collection = client.create_collection(
    name="research_paper",
    metadata={
        "department":"AI"
    }
)

# Real Production Structure

# Suppose you're building a company assistant.

# Bad:

# company_docs

# One giant collection.

# Better:

# hr_docs

# engineering_docs

# legal_docs

# finance_docs

# sales_docs

# Now retrieval becomes cleaner.

# Another Common Architecture

# Suppose you build:

# PDF Chatbot SaaS

# with multiple users.

# You might create:

# user_101_docs

# user_102_docs

# user_103_docs

# Separate collection per customer.

# This is very common.

## Collection Lifecycle

In [60]:
# client = chromadb.PersistentClient()

# collection = client.get_or_create_collection(
#     "docs"
# )

# collection.add(...)

# collection.query(...)

# collection.update(...)

# collection.delete(...)

# Lesson 8 — Metadata Filtering

This lesson is extremely important because it introduces one of the biggest concepts in production RAG systems:

Retrieval is not only about similarity. It is also about constraints.

What Is Metadata?

Metadata means:

Data about data.

Example document:

Employees receive 20 annual leaves.

Document itself:

Employees receive 20 annual leaves.

Metadata:

{
    "department": "HR",
    "year": 2025,
    "source": "employee_handbook.pdf"
}

## Metadata During Ingestion

In [ ]:
collection.add(
    documents=[
        "Employees get 20 annual leaves",
        "Q4 revenue increased by 15%"
    ],

    ids=[
        "doc1",
        "doc2"
    ],

    metadatas=[
        {
            "department": "HR"
        },

        {
            "department": "Finance"
        }
    ]
)

# Each document gets its own metadata.

## Understanding The Structure

Document 1:

{
    "text":
    "Employees get 20 annual leaves",

    "department":
    "HR"
}

Document 2:

{
    "text":
    "Q4 revenue increased by 15%",

    "department":
    "Finance"
}

## Query Without Filtering

In [ ]:
results = collection.query(
    query_texts=[
        "leave policy"
    ],
    n_results=5
)

# Searches:

# Everything

## Query With Filtering

In [ ]:
results = collection.query(
    query_texts=[
        "leave policy"
    ],

    where={
        "department": "HR"
    },

    n_results=5
)

# Now search space becomes:

# Only HR Documents


## NOTE THAT FILTERING HAPPENS BEFORE SIMILARITY SEARCH

# Lesson 9 — Building Mini Semantic Search Engine

## Step 2 — Create Database

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="./semantic_db"
)

# Recall

# PersistentClient means:

# Data survives program restart

# Very important in real applications.

## Step 3 — Create Collection

In [65]:
collection = client.get_or_create_collection(
    name="knowledge_base"
)

## Step 4 — Sample Documents

In [66]:
documents = [

    "Dogs are loyal pets",

    "Cats enjoy sleeping",

    "Python is a programming language",

    "Transformers power modern AI systems",

    "Neural networks learn from data",

    "Puppies are young dogs"
]

## Metadata

In [67]:
metadatas = [

    {"category":"animals"},

    {"category":"animals"},

    {"category":"programming"},

    {"category":"ai"},

    {"category":"ai"},

    {"category":"animals"}
]

## IDs

Every document requires a unique id.

In [68]:
ids = [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6"
]

## Step 5 — Add Documents

In [70]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print(collection.count())

6


## Step 6 — Search Function

In [71]:
def search(query):

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    return results["documents"][0]

## Test 1

In [72]:
print(
    search(
        "Tell me about puppies"
    )
)

['Puppies are young dogs', 'Dogs are loyal pets', 'Cats enjoy sleeping']


## Test 2

In [73]:
print(
    search(
        "Explain neural networks"
    )
)

['Neural networks learn from data', 'Transformers power modern AI systems', 'Puppies are young dogs']


## Adding Metadata Filtering

Now let's improve the system.

Suppose:

User only wants AI documents.

In [74]:
results = collection.query(
    query_texts=[
        "learning algorithms"
    ],

    where={
        "category":"ai"
    },

    n_results=3
)

## Reusable Filtered Search Function

In [75]:
def search_category(
    query,
    category
):

    results = collection.query(

        query_texts=[query],

        where={
            "category":category
        },

        n_results=3
    )

    return results["documents"][0]

Production Improvements

Our current engine has limitations.

Problem 1

Small Dataset

6 Documents

Real systems:

Thousands
Millions
Problem 2

No Chunking

Large PDFs need:

Chunk 1
Chunk 2
Chunk 3
...

We'll learn this in Phase 7.

Problem 3

No Reranking

Current:

Similarity Search

Production:

Similarity Search
      ↓
Reranker
      ↓
Final Results

Phase 7.

Problem 4

No LLM

Currently:

Retrieve Only

RAG:

Retrieve
+
Generate

Phase 7.